# Graph-Aware Retrieval for AI Agent Memory: Linking Memories Across Time

This notebook is the technical companion for Graph- Aware retrieval Article in the Oracle AI Agent Memory. It shows how memory links and graph-aware retrieval help an agent follow facts as they change, instead of treating every memory as a separate item with no history.

The walkthrough uses a support case where a customer's delivery preference changes over time. We first store the original preference, then add newer memories that replace, sharpen, or support that earlier context. After that, we compare direct retrieval with graph-aware retrieval so you can see how `linked_results` gives the agent relationship context for a future response.

This notebook stays focused on memory evolution. Image memory, observability, benchmark numbers, and broader enterprise positioning belong in the other release assets.

## What This Notebook Demonstrates

By the end of the notebook, you will have a compact graph-memory example that covers the Article 2 requirements:

- create initial durable memories for a support scenario;
- add a newer memory that `supersedes` an older memory;
- add `refines` and `supports` examples;
- explain the full typed-link vocabulary, including `duplicates` and `contradicts`;
- distinguish current `valid` memories from older `invalid` or historical memories;
- compare normal memory search with graph-aware retrieval;
- use `num_hops=0` and `num_hops=1`;
- inspect `linked_results`;
- show why older memories can remain useful context even when they should not drive the current answer.

The runnable path demonstrates `supersedes`, `refines`, and `supports` because those relationships fit the support story cleanly. `duplicates` and `contradicts` are included as conceptual link types so the notebook still reflects the complete Article 2 framing without adding artificial records.

## Release Validation Note

This notebook targets Oracle AI Agent Memory 26.8 graph-aware retrieval. If you run it with an earlier package, the setup cells may stop at the API check because methods such as `link_records` or search parameters such as `num_hops`, `max_linked_results`, and `include_invalid_results` are release-specific.

For draft validation, it is still useful to run the connection and package-check cells. They confirm whether the database, Python environment, and package version are ready before the graph-memory workflow is executed.

## Conceptual Flow

A flat memory search answers one question: "Which stored memories look similar to this query?" That is useful, but it is not enough when facts evolve.

Graph-aware retrieval adds a second question: "What related memories explain this result?" The answer can include an older fact that was superseded, a more precise memory that refines a broad one, or a supporting fact from a workflow event.

**Memory creation**

`Support event` -> `Durable memory record` -> `Lifecycle state`

**Memory linking**

`Newer or related record` -> `Typed relationship` -> `Earlier or related record`

**Graph-aware retrieval**

`User query` -> `Direct matches` -> `Bounded graph expansion` -> `Result + linked_results`

In this notebook, `num_hops=0` means direct retrieval only. `num_hops=1` means the search can include one step of linked memory context around the direct result.

## Link Types and Lifecycle States

Graph-aware memory has two related ideas: relationship type and lifecycle state.

| Concept | Meaning | Example in an agent memory workflow |
| --- | --- | --- |
| `supersedes` | A newer memory replaces an older one for future use. | Afternoon delivery supersedes a previous morning-delivery preference. |
| `refines` | A newer memory adds precision to an earlier one. | "Afternoon delivery" is refined to "2 PM to 5 PM." |
| `supports` | One memory provides evidence or operational context for another. | A replacement-shipment record supports the current delivery preference. |
| `duplicates` | Two memories represent the same or nearly the same fact. | Two extracted preferences repeat the same delivery instruction. |
| `contradicts` | Two memories conflict and may need review or resolution. | One memory says afternoon delivery, another says morning only. |
| `valid` | The memory is eligible to guide current behavior. | The latest delivery preference is valid. |
| `invalid` | The memory should not drive the current answer, but may remain useful historically. | The superseded morning preference is invalid or historical. |

The notebook demonstrates `supersedes`, `refines`, and `supports` in code. It describes `duplicates` and `contradicts` so the full link model is represented without making the support scenario unnecessarily noisy.

In [ ]:
link_type_reference = pd.DataFrame(
    [
        {"link_type": "supersedes", "used_in_demo": True, "purpose": "newer memory replaces an older memory"},
        {"link_type": "refines", "used_in_demo": True, "purpose": "newer memory makes an earlier memory more precise"},
        {"link_type": "supports", "used_in_demo": True, "purpose": "one memory provides evidence or workflow context for another"},
        {"link_type": "duplicates", "used_in_demo": False, "purpose": "two memories represent the same or nearly the same fact"},
        {"link_type": "contradicts", "used_in_demo": False, "purpose": "two memories conflict and may need resolution"},
    ]
)

lifecycle_reference = pd.DataFrame(
    [
        {"state": "valid", "role": "eligible to guide current agent behavior"},
        {"state": "invalid", "role": "not the current answer, but useful as linked historical context"},
    ]
)

display(link_type_reference)
display(lifecycle_reference)

## Part 1 - Install Packages

Run this once in a fresh Python environment. If the package was already imported in the current kernel, restart the kernel after installation and run the notebook from the beginning.

In [ ]:
# Run this cell once in a clean environment.
# Restart the kernel if any package is upgraded.
%pip install --upgrade oracleagentmemory oracledb pandas python-dotenv --quiet --disable-pip-version-check

## Part 2 - Configure the Runtime

Set the database and model values as environment variables or in a private `.env` file next to the notebook.

Required values:

```text
DB_USER=<database user>
DB_PASSWORD=<database password>
DB_DSN=<database DSN, service name, or connect descriptor>
MODEL_PROVIDER_API_KEY=<model provider API key>
```

Optional values:

```text
DB_WALLET_LOCATION=<path-to-unzipped-adb-wallet>
MEMORY_LLM_MODEL=gpt-4o-mini
MEMORY_EMBEDDING_MODEL=text-embedding-3-small
MEMORY_EMBEDDING_DIMENSION=1536
OAMP_MEMORY_STORE_ID=GRAPHMEMORYDEMO
```

For FreeSQL, open **Connect to the Database**, choose **Python**, then copy the username, password, and DSN into the environment. For Autonomous AI Database, use the service name from the wallet/client credentials and set `DB_WALLET_LOCATION` only if your runtime requires it.

In [ ]:
import inspect
import os
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

import oracledb
import pandas as pd

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

oracledb.defaults.program = "devrel-developerhub-graph-aware-retrieval-agent-memory"

print("Runtime imports: READY")
print("Oracle Database program identifier: READY")

In [ ]:
import importlib.metadata as metadata

package_version = metadata.version("oracleagentmemory")
version_parts = tuple(int(part) for part in package_version.split(".")[:2])

print(f"oracleagentmemory package version: {package_version}")
if version_parts < (26, 8):
    raise RuntimeError("Install oracleagentmemory 26.8 or later to run graph-aware retrieval examples.")

In [ ]:
def read_config():
    config = {
        "DB_USER": os.getenv("DB_USER") or os.getenv("ORACLE_USER"),
        "DB_PASSWORD": os.getenv("DB_PASSWORD") or os.getenv("ORACLE_PASSWORD"),
        "DB_DSN": os.getenv("DB_DSN") or os.getenv("ORACLE_DSN"),
        "DB_WALLET_LOCATION": os.getenv("DB_WALLET_LOCATION") or os.getenv("TNS_ADMIN"),
        "MODEL_PROVIDER_API_KEY": os.getenv("MODEL_PROVIDER_API_KEY") or os.getenv("OPENAI_API_KEY"),
        "MEMORY_LLM_MODEL": os.getenv("MEMORY_LLM_MODEL", "gpt-4o-mini"),
        "MEMORY_EMBEDDING_MODEL": os.getenv("MEMORY_EMBEDDING_MODEL") or os.getenv("EMBED_MODEL") or "text-embedding-3-small",
        "MEMORY_EMBEDDING_DIMENSION": int(os.getenv("MEMORY_EMBEDDING_DIMENSION") or os.getenv("EMBED_DIM") or "1536"),
        "MEMORY_STORE_ID": os.getenv("OAMP_MEMORY_STORE_ID", "GRAPHMEMORYDEMO"),
    }

    required = ["DB_USER", "DB_PASSWORD", "DB_DSN", "MODEL_PROVIDER_API_KEY"]
    missing = [name for name in required if not config.get(name)]
    if missing:
        raise RuntimeError("Missing required configuration values: " + ", ".join(missing))
    return config


CONFIG = read_config()
print("Configuration: READY")

## Part 3 - Connect to Oracle AI Database

This notebook can be validated against FreeSQL, Autonomous AI Database, Oracle AI Database, or a local Oracle Free container. For public release positioning, keep the hosted Oracle Database path visible. For local notebook validation, the same connection variables can point to Podman Oracle Free.

The preflight below creates and drops a small table. That confirms the connection and table permissions needed for package-managed memory objects without printing credentials.

In [ ]:
pool_kwargs = {
    "user": CONFIG["DB_USER"],
    "password": CONFIG["DB_PASSWORD"],
    "dsn": CONFIG["DB_DSN"],
    "min": 1,
    "max": 4,
    "increment": 1,
}

if CONFIG["DB_WALLET_LOCATION"]:
    pool_kwargs["config_dir"] = CONFIG["DB_WALLET_LOCATION"]

db_pool = oracledb.create_pool(**pool_kwargs)
test_table = f"OAM_GRAPH_CHECK_{uuid4().hex[:8].upper()}"

try:
    with db_pool.acquire() as connection:
        with connection.cursor() as cursor:
            cursor.execute("SELECT 1 FROM dual").fetchone()
            cursor.execute(
                f'''
                CREATE TABLE {test_table} (
                    id NUMBER PRIMARY KEY,
                    note VARCHAR2(100)
                )
                '''
            )
            cursor.execute(
                f"INSERT INTO {test_table} (id, note) VALUES (:id, :note)",
                id=1,
                note="graph memory permission check",
            )
            row_count = cursor.execute(f"SELECT COUNT(*) FROM {test_table}").fetchone()[0]
            if row_count != 1:
                raise RuntimeError("Table permission check returned an unexpected row count.")
        connection.commit()
finally:
    try:
        with db_pool.acquire() as cleanup_connection:
            with cleanup_connection.cursor() as cursor:
                cursor.execute(f"DROP TABLE {test_table} PURGE")
            cleanup_connection.commit()
    except oracledb.Error:
        pass

print("Oracle AI Database connection: READY")
print("Table permissions: READY")

## Part 4 - Build the Memory Client

This section creates the LLM, embedder, and database-backed memory store. The important release detail is the schema policy: graph-aware retrieval depends on managed schema support for memory records, memory links, lifecycle state, and graph traversal.

For a fresh notebook environment, `SchemaPolicy.CREATE_IF_NECESSARY` is convenient because it lets the package create or upgrade its managed objects. For shared or production environments, teams should normally run schema changes deliberately and use a stricter policy after migration.

In [ ]:
from oracleagentmemory.core import OracleAgentMemory, OracleDBMemoryStore, SchemaPolicy
from oracleagentmemory.core import MemoryExtractionConfig, SearchStrategy
from oracleagentmemory.core.embedders import Embedder
from oracleagentmemory.core.llms import Llm, LlmApiType


llm_kwargs = {
    "model": CONFIG["MEMORY_LLM_MODEL"],
    "api_key": CONFIG["MODEL_PROVIDER_API_KEY"],
    "temperature": 0,
    "max_tokens": 2_000,
}
if os.getenv("MEMORY_LLM_API_BASE"):
    llm_kwargs["api_base"] = os.getenv("MEMORY_LLM_API_BASE")
if os.getenv("MEMORY_LLM_API_TYPE", "chat_completions") == "responses":
    llm_kwargs["api_type"] = LlmApiType.RESPONSES

memory_llm = Llm(**llm_kwargs)

embedder = Embedder(
    model=CONFIG["MEMORY_EMBEDDING_MODEL"],
    api_key=CONFIG["MODEL_PROVIDER_API_KEY"],
    embedding_dimension=CONFIG["MEMORY_EMBEDDING_DIMENSION"],
    max_input_tokens=512,
    normalize=True,
)

store = OracleDBMemoryStore(
    pool=db_pool,
    embedder=embedder,
    memory_store_id=CONFIG["MEMORY_STORE_ID"],
    schema_policy=SchemaPolicy.CREATE_IF_NECESSARY,
    search_strategy=SearchStrategy.VECTOR,
    vector_dim=CONFIG["MEMORY_EMBEDDING_DIMENSION"],
)

memory = OracleAgentMemory(
    store=store,
    llm=memory_llm,
    memory_extraction_config=MemoryExtractionConfig(
        memory_extraction_frequency=1,
        enable_context_summary=False,
    ),
)

print("Database-backed memory client: READY")
print("Managed schema policy: CREATE_IF_NECESSARY")

In [ ]:
def has_parameter(callable_obj, parameter_name):
    try:
        return parameter_name in inspect.signature(callable_obj).parameters
    except (TypeError, ValueError):
        return False


api_checks = pd.DataFrame(
    [
        {"Capability": "Create explicit record links", "Check": "OracleAgentMemory.link_records", "Available": hasattr(memory, "link_records")},
        {"Capability": "Search with graph hops", "Check": "search(..., num_hops=...)", "Available": has_parameter(memory.search, "num_hops")},
        {"Capability": "Limit linked results", "Check": "search(..., max_linked_results=...)", "Available": has_parameter(memory.search, "max_linked_results")},
        {"Capability": "Control invalid top-level results", "Check": "search(..., include_invalid_results=...)", "Available": has_parameter(memory.search, "include_invalid_results")},
    ]
)

display(api_checks)

if not api_checks["Available"].all():
    missing = ", ".join(api_checks.loc[~api_checks["Available"], "Capability"])
    raise RuntimeError("Install a graph-aware oracleagentmemory release. Missing capabilities: " + missing)

print("Graph-aware retrieval APIs: READY")

## Part 5 - Create Evolving Memories

The support case starts with four durable memories:

- an older delivery preference;
- a newer preference that replaces the older one;
- a more precise delivery-window detail;
- a replacement-shipment fact that supports the current preference.

The metadata includes a simple `state` value so the notebook output is easy to read. In the product model, memory lifecycle states such as `valid` and `invalid` help distinguish memories that should guide current behavior from memories that mainly explain history.

In [ ]:
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")
USER_ID = f"customer_graph_{RUN_ID}"
AGENT_ID = "support-agent"

seed_memories = [
    {
        "label": "original_preference",
        "record_type": "preference",
        "text": "Customer prefers morning delivery windows for replacement shipments.",
        "metadata": {"tenant": "demo", "case_id": "CASE-8421", "topic": "delivery_preference", "state": "historical"},
    },
    {
        "label": "updated_preference",
        "record_type": "preference",
        "text": "Customer now prefers afternoon delivery windows for replacement shipments because mornings conflict with work.",
        "metadata": {"tenant": "demo", "case_id": "CASE-8421", "topic": "delivery_preference", "state": "current"},
    },
    {
        "label": "refined_preference",
        "record_type": "preference",
        "text": "For replacement shipments, the best delivery window is 2 PM to 5 PM local time.",
        "metadata": {"tenant": "demo", "case_id": "CASE-8421", "topic": "delivery_preference", "state": "current"},
    },
    {
        "label": "replacement_context",
        "record_type": "fact",
        "text": "Replacement shipment RMA-8842 is tied to order ORD-7421 and should use the current delivery-window preference.",
        "metadata": {"tenant": "demo", "case_id": "CASE-8421", "topic": "replacement", "state": "current"},
    },
]

pd.DataFrame(seed_memories)

In [ ]:
created = {}

for item in seed_memories:
    result = memory.add_memory(
        user_id=USER_ID,
        agent_id=AGENT_ID,
        text=item["text"],
        metadata=item["metadata"],
    )
    created[item["label"]] = result


def get_memory_id(label):
    result = created[label]
    value = getattr(result, "memory_id", getattr(result, "id", None))
    if value is None:
        raise RuntimeError(f"Could not read memory id for {label} from add_memory result.")
    return value


def get_record_type(label):
    for item in seed_memories:
        if item["label"] == label:
            return item["record_type"]
    raise KeyError(label)


memory_table = pd.DataFrame(
    [
        {
            "label": label,
            "record_type": get_record_type(label),
            "memory_id": get_memory_id(label),
            "text": item["text"],
            "state": item["metadata"]["state"],
        }
        for label, item in ((item["label"], item) for item in seed_memories)
    ]
)

display(memory_table)
print("Durable memories: READY")

## Part 6 - Add Typed Memory Links

Memory links make relationships explicit. This notebook creates three links:

- `updated_preference` **supersedes** `original_preference`;
- `refined_preference` **refines** `updated_preference`;
- `replacement_context` **supports** `refined_preference`.

The broader link vocabulary also includes `duplicates` and `contradicts`. Those are useful when a memory repeats another record or when two records cannot both be true. They are not created in this scenario because the support case is meant to stay clean and easy to inspect.

The key lifecycle point is that older records do not have to disappear. A superseded or invalid memory can stay available as historical context while the newer valid memory drives the response.

In [ ]:
links_to_create = [
    {
        "source_label": "updated_preference",
        "target_label": "original_preference",
        "relation_type": "supersedes",
        "reason": "The customer's delivery-window preference changed from morning to afternoon.",
    },
    {
        "source_label": "refined_preference",
        "target_label": "updated_preference",
        "relation_type": "refines",
        "reason": "The newer preference is narrowed to a specific afternoon window.",
    },
    {
        "source_label": "replacement_context",
        "target_label": "refined_preference",
        "relation_type": "supports",
        "reason": "The replacement shipment should follow the current delivery-window preference.",
    },
]

created_links = []
for link in links_to_create:
    created_link = memory.link_records(
        source_record_id=get_memory_id(link["source_label"]),
        source_record_type=get_record_type(link["source_label"]),
        target_record_id=get_memory_id(link["target_label"]),
        target_record_type=get_record_type(link["target_label"]),
        relation_type=link["relation_type"],
        metadata={"reason": link["reason"], "created_by": "notebook"},
    )
    created_links.append(created_link)

link_table = pd.DataFrame(
    [
        {
            "relationship": link["relation_type"],
            "from_memory": link["source_label"],
            "to_memory": link["target_label"],
            "why": link["reason"],
        }
        for link in links_to_create
    ]
)

display(link_table)
print("Typed memory links: READY")

## Part 7 - Compare Direct and Graph-Aware Retrieval

The same question is searched two ways:

- `num_hops=0` keeps retrieval flat and returns only direct matches.
- `num_hops=1` lets the search bring back one-hop relationship context through memory links.

`max_linked_results` keeps the expansion bounded. `include_invalid_results=False` is useful when invalid or historical memories should not appear as top-level answers, while linked historical context can still explain how the current fact came to be.

In [ ]:
def search_memories(query, num_hops=0, max_linked_results=3, include_invalid_results=True):
    return memory.search(
        query=query,
        user_id=USER_ID,
        agent_id=AGENT_ID,
        num_hops=num_hops,
        max_linked_results=max_linked_results,
        include_invalid_results=include_invalid_results,
    )


query = "What delivery window should I use for replacement shipment RMA-8842?"
direct_results = search_memories(query, num_hops=0, include_invalid_results=False)
graph_results = search_memories(query, num_hops=1, max_linked_results=5, include_invalid_results=True)

print("Direct search: READY")
print("Graph-aware search: READY")

In [ ]:
def result_text(result):
    return getattr(result, "text", getattr(result, "memory", getattr(result, "content", "")))


def result_score(result):
    return getattr(result, "score", getattr(result, "relevance_score", None))


def linked_results(result):
    return getattr(result, "linked_results", []) or []


def summarize_results(results, label):
    rows = []
    for rank, result in enumerate(results, start=1):
        rows.append(
            {
                "search_mode": label,
                "rank": rank,
                "score": result_score(result),
                "memory": result_text(result),
                "linked_result_count": len(linked_results(result)),
            }
        )
    return pd.DataFrame(rows)


comparison = pd.concat(
    [
        summarize_results(direct_results, "num_hops=0"),
        summarize_results(graph_results, "num_hops=1"),
    ],
    ignore_index=True,
)

display(comparison)

## Part 8 - Inspect `linked_results`

`linked_results` is the bridge from graph storage to agent behavior. The direct result tells the agent what matched the query. The linked results explain nearby context: what was superseded, what refined the answer, or what supporting fact makes the answer reliable.

That separation matters because applications can decide how much relational context to place into the next prompt instead of blindly sending every related memory.

### Read the Search Difference

The comparison table is meant to make the retrieval behavior visible:

- `num_hops=0` shows what the agent gets from ordinary direct memory search.
- `num_hops=1` shows what changes when the search can expand through one memory-link hop.

In a blog screenshot, the most useful signal is the `linked_result_count` column. A nonzero value means the direct result brought relationship context with it.

In [ ]:
def explain_search_difference(direct_results, graph_results):
    direct_link_count = sum(len(linked_results(result)) for result in direct_results)
    graph_link_count = sum(len(linked_results(result)) for result in graph_results)

    print("Retrieval takeaway")
    print(f"Direct search returned {len(direct_results)} direct result(s) and {direct_link_count} linked result(s).")
    print(f"Graph-aware search returned {len(graph_results)} direct result(s) and {graph_link_count} linked result(s).")

    if graph_link_count > direct_link_count:
        print("Graph-aware retrieval added relationship context that direct search did not include.")
    elif graph_link_count:
        print("Graph-aware retrieval returned linked context; inspect linked_results for relationship details.")
    else:
        print("No linked context was returned for this query. Try a query closer to the linked memories or increase max_linked_results.")


explain_search_difference(direct_results, graph_results)

In [ ]:
linked_rows = []

for parent_rank, result in enumerate(graph_results, start=1):
    for linked_rank, linked in enumerate(linked_results(result), start=1):
        linked_rows.append(
            {
                "parent_rank": parent_rank,
                "linked_rank": linked_rank,
                "relationship": getattr(linked, "link_type", getattr(linked, "relationship", "linked")),
                "linked_memory": result_text(linked),
            }
        )

if linked_rows:
    display(pd.DataFrame(linked_rows))
else:
    print("No linked results returned for this query. Try increasing num_hops or max_linked_results.")

## Part 9 - Historical Context and Current Answers

Memory lifecycle is not only about deleting stale facts. A memory can stop being the best current answer and still remain valuable as context.

In the example, the morning-delivery preference is historical after the customer changes to afternoon delivery. If the product marks that older record `invalid`, graph-aware retrieval can still use it to explain the change while keeping the current valid preference front and center.

### Build Prompt-Ready Graph Context

This cell formats the top graph-aware result as compact context an application could pass to an agent. It keeps the current memory separate from related context, which helps the prompt use the latest valid fact while still seeing useful history.

In [ ]:
def format_prompt_context(results):
    if not results:
        return "No graph-aware memory context was returned."

    top_result = results[0]
    lines = [
        "Memory context for the next agent response",
        "==========================================",
        "",
        "Current or directly matched memory:",
        f"- {result_text(top_result)}",
    ]

    nearby = linked_results(top_result)
    if nearby:
        lines.extend(["", "Linked context:"])
        for linked in nearby:
            relationship = getattr(linked, "link_type", getattr(linked, "relationship", "linked"))
            lines.append(f"- {relationship}: {result_text(linked)}")
    else:
        lines.extend(["", "Linked context:", "- No linked context returned for the top result."])

    return "\n".join(lines)


print(format_prompt_context(graph_results))

In [ ]:
historical_query = "Did the customer ever prefer morning delivery?"
historical_context = search_memories(
    historical_query,
    num_hops=1,
    max_linked_results=5,
    include_invalid_results=True,
)

display(summarize_results(historical_context, "historical context search"))

In [ ]:
if graph_results:
    top_result = graph_results[0]
    print("Direct memory:")
    print(result_text(top_result))

    nearby = linked_results(top_result)
    if nearby:
        print("\nLinked context:")
        for linked in nearby:
            relationship = getattr(linked, "link_type", getattr(linked, "relationship", "linked"))
            print(f"- {relationship}: {result_text(linked)}")
    elif hasattr(top_result, "format_content"):
        print(top_result.format_content())
else:
    print("No graph-aware results to render.")

## Optional - Automatic Linking During Extraction

Use this section only if your team wants the Article 2 notebook to show automatic linking. The explicit-link workflow above is the stable core demo because it uses relationships the application already knows.

Automatic linking can be enabled through `MemoryExtractionConfig.memory_link_extraction_mode` when the installed package exposes `MemoryLinkExtractionMode`. Exact outputs can vary because extraction and linking are model-assisted.

### Keep Retrieval Scoped

Graph-aware retrieval should still run inside application boundaries. The searches in this notebook pass `user_id` and `agent_id`, and the stored memories include metadata such as tenant, case, topic, and state. In a real application, those values help decide which memories are eligible before graph expansion adds linked context.

In [ ]:
scope_rows = [
    {"scope_field": "user_id", "example_value": USER_ID, "why_it_matters": "keeps retrieval tied to the current user or actor"},
    {"scope_field": "agent_id", "example_value": AGENT_ID, "why_it_matters": "keeps retrieval tied to the agent/application context"},
    {"scope_field": "metadata.tenant", "example_value": "demo", "why_it_matters": "supports tenant or workspace boundaries"},
    {"scope_field": "metadata.case_id", "example_value": "CASE-8421", "why_it_matters": "keeps support-case context narrow"},
]

display(pd.DataFrame(scope_rows))

In [ ]:
def enum_value(enum_class, preferred_names):
    for name in preferred_names:
        if hasattr(enum_class, name):
            return getattr(enum_class, name)
    return None


try:
    from oracleagentmemory.core import MemoryLinkExtractionMode
except Exception:
    MemoryLinkExtractionMode = None

if MemoryLinkExtractionMode is None:
    print("Automatic linking enum is not available in this installed package version.")
else:
    link_mode = enum_value(MemoryLinkExtractionMode, ["POST_EXTRACTION"])
    auto_link_config = MemoryExtractionConfig(
        memory_extraction_frequency=1,
        enable_context_summary=False,
        memory_link_extraction_mode=link_mode,
    )
    auto_memory = OracleAgentMemory(
        store=store,
        llm=memory_llm,
        memory_extraction_config=auto_link_config,
    )
    print("Automatic linking extraction config: READY")
    print("Selected mode:", link_mode)

In [ ]:
if MemoryLinkExtractionMode is not None:
    auto_thread = auto_memory.create_thread(
        thread_id=f"auto_graph_support_{RUN_ID}",
        user_id=USER_ID,
        agent_id=AGENT_ID,
    )

    await auto_thread.add_messages_async(
        [
            {
                "role": "user",
                "content": "For order ORD-7421, please stop using morning delivery. Afternoon is now the right delivery window.",
            },
            {
                "role": "assistant",
                "content": "I will remember that order ORD-7421 should use afternoon delivery going forward.",
            },
            {
                "role": "user",
                "content": "This is for replacement shipment RMA-8842, tied to the same case.",
            },
        ],
        metadata={"tenant": "demo", "case_id": "CASE-8421", "source": "support_chat"},
    )

    await auto_thread.wait_for_memory_extraction_async()
    print("Automatic-linking extraction thread: READY")
else:
    print("Skipped because automatic linking is not available in this package version.")

## Output Review Checklist

After running the notebook, confirm that it matches the Article 2 brief:

- durable memories are created for an evolving support case;
- a newer memory `supersedes` an older memory;
- `refines` and `supports` links are demonstrated;
- `duplicates` and `contradicts` are described as part of the typed-link vocabulary;
- current `valid` memory and older `invalid` or historical memory are explained;
- direct search with `num_hops=0` returns normal memory matches;
- graph-aware search with `num_hops=1` exposes `linked_results`;
- older memories remain available as context without replacing the current answer.

## Cleanup

This closes the database pool. It does not delete the package-managed memory records, so you can inspect them after the run if needed.

In [ ]:
try:
    db_pool.close(force=True)
except Exception:
    pass

print("Cleanup and shutdown: READY")

## Summary

Graph-aware retrieval lets Oracle AI Agent Memory model memory as something that evolves. Instead of storing isolated facts forever, an application can connect memories with relationships such as `supersedes`, `refines`, `supports`, `duplicates`, and `contradicts`.

The takeaway is straightforward: agents need current facts, but they also need enough history to understand why those facts changed. `num_hops` controls when retrieval expands beyond direct matches, and `linked_results` gives future agent responses a compact way to use that relationship context.